In [43]:
import os

os.chdir('/Users/progressive/Documents/Courses/Final-Year-Disease-Prediction')

print("Current directory:", os.getcwd())
print("\nFiles in data/raw/:")
print(os.listdir('data/raw/'))
print("\nFiles in data/processed/:")
print(os.listdir('data/processed/'))

Current directory: /Users/progressive/Documents/Courses/Final-Year-Disease-Prediction

Files in data/raw/:
['era5_nigeria_full.nc', '.DS_Store', '92024a5dcd333d6c7e2215bffedf1842.grib', '92024a5dcd333d6c7e2215bffedf1842.grib.5b7b6.idx', 'era5_nigeria_v2.nc', 'nigeria_cholera.csv']

Files in data/processed/:
['final_training_data.csv', 'era5_nigeria_v2_monthly.csv', 'era5_nigeria_monthly.csv']


In [1]:
import requests
import pandas as pd

# WHO GHO API — Cholera cases for Nigeria
url = "https://ghoapi.azureedge.net/api/CHOLERA_0000000001"

response = requests.get(url)
data = response.json()

df_cholera = pd.DataFrame(data['value'])

# Filter Nigeria only
nigeria_cholera = df_cholera[df_cholera['SpatialDim'] == 'NGA'].copy()

# Keep useful columns
nigeria_cholera = nigeria_cholera[['TimeDim', 'NumericValue']].copy()
nigeria_cholera.columns = ['year', 'cholera_cases']
nigeria_cholera['year'] = nigeria_cholera['year'].astype(int)
nigeria_cholera = nigeria_cholera.sort_values('year').reset_index(drop=True)

print("Nigeria Cholera data:")
print(nigeria_cholera)
print("\nYears available:", sorted(nigeria_cholera['year'].unique()))

Nigeria Cholera data:
    year  cholera_cases
0   1970            NaN
1   1971            NaN
2   1972            NaN
3   1973            NaN
4   1975            NaN
5   1976            NaN
6   1977            NaN
7   1978            NaN
8   1979            NaN
9   1980            NaN
10  1981            NaN
11  1982            NaN
12  1983            NaN
13  1984            NaN
14  1985            NaN
15  1986            NaN
16  1987            NaN
17  1988            NaN
18  1989            NaN
19  1991            NaN
20  1992            NaN
21  1993            NaN
22  1994            NaN
23  1995            NaN
24  1996            NaN
25  1997            NaN
26  1998            NaN
27  1999            NaN
28  2000            NaN
29  2001            NaN
30  2002            NaN
31  2003            NaN
32  2004            NaN
33  2005            NaN
34  2006            NaN
35  2007            NaN
36  2008            NaN
37  2009            NaN
38  2010            NaN
39  2011          

In [2]:
# Filter to 2010-2016 only (what WHO has)
cholera_available = nigeria_cholera[nigeria_cholera['year'] >= 2010].copy()
print("Available 2010+:")
print(cholera_available)

Available 2010+:
    year  cholera_cases
38  2010            NaN
39  2011            NaN
40  2012          597.0
41  2013         6600.0
42  2014        35996.0
43  2015         5290.0
44  2016          768.0


In [4]:
# Manually add 2017-2025 from published NCDC annual reports
cholera_extra = pd.DataFrame({
    'year': [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'cholera_cases': [1956, 3640, 30407, 2558, 112746, 48574, 18088, 8173, None]
})

# Combine both
nigeria_cholera_full = pd.concat([cholera_available, cholera_extra], ignore_index=True)
nigeria_cholera_full = nigeria_cholera_full.sort_values('year').reset_index(drop=True)

print("Full cholera dataset 2010-2025:")
print(nigeria_cholera_full)

# Save it
nigeria_cholera_full.to_csv('../data/raw/nigeria_cholera.csv', index=False)
print("\n✅ Saved!")


Full cholera dataset 2010-2025:
    year  cholera_cases
0   2010            NaN
1   2011            NaN
2   2012          597.0
3   2013         6600.0
4   2014        35996.0
5   2015         5290.0
6   2016          768.0
7   2017         1956.0
8   2018         3640.0
9   2019        30407.0
10  2020         2558.0
11  2021       112746.0
12  2022        48574.0
13  2023        18088.0
14  2024         8173.0
15  2025            NaN

✅ Saved!


In [6]:
# Fill 2010, 2011 from published WHO reports
# Fill 2025 with 2024 trend (partial year estimate)
nigeria_cholera_full.loc[nigeria_cholera_full['year'] == 2010, 'cholera_cases'] = 41787.0
nigeria_cholera_full.loc[nigeria_cholera_full['year'] == 2011, 'cholera_cases'] = 23366.0
nigeria_cholera_full.loc[nigeria_cholera_full['year'] == 2025, 'cholera_cases'] = 5000.0  # estimate

# Verify
print("Cleaned cholera data:")
print(nigeria_cholera_full)
print("\nAny missing values:", nigeria_cholera_full.isnull().sum().sum())

# Overwrite saved file
nigeria_cholera_full.to_csv('../data/raw/nigeria_cholera.csv', index=False)
print("✅ Saved!")

Cleaned cholera data:
    year  cholera_cases
0   2010        41787.0
1   2011        23366.0
2   2012          597.0
3   2013         6600.0
4   2014        35996.0
5   2015         5290.0
6   2016          768.0
7   2017         1956.0
8   2018         3640.0
9   2019        30407.0
10  2020         2558.0
11  2021       112746.0
12  2022        48574.0
13  2023        18088.0
14  2024         8173.0
15  2025         5000.0

Any missing values: 0
✅ Saved!


In [13]:
import os

os.chdir('/Users/progressive/Documents/Courses/Final-Year-Disease-Prediction')

print("Files in data/raw:")
for f in os.listdir('data/raw'):
    print(f)

Files in data/raw:
era5_nigeria_full.nc
92024a5dcd333d6c7e2215bffedf1842.grib
92024a5dcd333d6c7e2215bffedf1842.grib.5b7b6.idx
nigeria_cholera.csv
clean_lassa_fever_map_data.csv


In [33]:
import xarray as xr
import pandas as pd
import numpy as np

# Load dataset
ds = xr.open_dataset('data/raw/era5_nigeria_v2.nc', engine='netcdf4')

# Check time range
print("Time range:", ds.valid_time.values[0], "to", ds.valid_time.values[-1])
print("Total months:", len(ds.valid_time))

# Convert to dataframe
df_raw = ds.to_dataframe().reset_index()
print("\nShape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
print(df_raw.head(3))

Time range: 2008-01-01T00:00:00.000000000 to 2026-04-01T00:00:00.000000000
Total months: 220

Shape: (441980, 8)
Columns: ['valid_time', 'latitude', 'longitude', 'u10', 'd2m', 't2m', 'number', 'expver']
  valid_time  latitude  longitude       u10         d2m         t2m  number  \
0 2008-01-01      14.0       3.00 -2.744750  271.428223  293.573486       0   
1 2008-01-01      14.0       3.25 -2.749633  271.502441  293.479736       0   
2 2008-01-01      14.0       3.50 -2.768188  271.486816  293.440674       0   

  expver  
0   0001  
1   0001  
2   0001  


In [38]:
import os
os.chdir('/Users/progressive/Documents/Courses/Final-Year-Disease-Prediction')
print("✅ Directory:", os.getcwd())

✅ Directory: /Users/progressive/Documents/Courses/Final-Year-Disease-Prediction


In [34]:
# Drop unneeded columns
df_raw = df_raw.drop(columns=['number', 'expver'])

# Convert temperature and dewpoint from Kelvin to Celsius
df_raw['temperature_c'] = df_raw['t2m'] - 273.15
df_raw['dewpoint_c']    = df_raw['d2m'] - 273.15

# Calculate relative humidity from dewpoint
df_raw['humidity_pct'] = 100 * (
    np.exp((17.625 * df_raw['dewpoint_c']) / (243.04 + df_raw['dewpoint_c'])) /
    np.exp((17.625 * df_raw['temperature_c']) / (243.04 + df_raw['temperature_c']))
)

# Keep wind speed (u10 = eastward wind component m/s)
# Convert to absolute wind speed
df_raw['wind_speed'] = df_raw['u10'].abs()

# Drop original Kelvin columns
df_raw = df_raw.drop(columns=['t2m', 'd2m', 'u10'])

# Extract time fields
df_raw['valid_time'] = pd.to_datetime(df_raw['valid_time'])
df_raw['year']       = df_raw['valid_time'].dt.year
df_raw['month']      = df_raw['valid_time'].dt.month

# Aggregate to national monthly averages
df_weather_v2 = df_raw.groupby(['year', 'month']).agg(
    temperature_c = ('temperature_c', 'mean'),
    humidity_pct  = ('humidity_pct',  'mean'),
    wind_speed    = ('wind_speed',    'mean'),
).reset_index()

print("✅ New weather dataset shape:", df_weather_v2.shape)
print("Years:", df_weather_v2['year'].min(), "to", df_weather_v2['year'].max())
print(df_weather_v2.head(12))

# Save
df_weather_v2.to_csv('data/processed/era5_nigeria_v2_monthly.csv', index=False)
print("✅ Saved to data/processed/era5_nigeria_v2_monthly.csv")

✅ New weather dataset shape: (220, 5)
Years: 2008 to 2026
    year  month  temperature_c  humidity_pct  wind_speed
0   2008      1      23.870647     35.920704    1.263670
1   2008      2      25.802450     29.349182    1.775623
2   2008      3      28.990774     40.332932    1.191437
3   2008      4      28.880919     49.625439    0.864664
4   2008      5      28.485775     62.815266    0.713356
5   2008      6      27.099874     71.180832    1.055518
6   2008      7      25.390240     78.898163    1.554146
7   2008      8      24.813053     82.868454    1.367703
8   2008      9      25.605160     80.295982    0.941896
9   2008     10      26.397943     66.530289    0.778065
10  2008     11      26.207600     48.486927    1.297227
11  2008     12      25.674116     44.154854    1.146068
✅ Saved to data/processed/era5_nigeria_v2_monthly.csv


In [ ]:
# Drop unneeded columns
df_raw = df_raw.drop(columns=['number', 'expver'])

# Convert temperature and dewpoint from Kelvin to Celsius
df_raw['temperature_c'] = df_raw['t2m'] - 273.15
df_raw['dewpoint_c']    = df_raw['d2m'] - 273.15

# Calculate relative humidity from dewpoint
df_raw['humidity_pct'] = 100 * (
    np.exp((17.625 * df_raw['dewpoint_c']) / (243.04 + df_raw['dewpoint_c'])) /
    np.exp((17.625 * df_raw['temperature_c']) / (243.04 + df_raw['temperature_c']))
)

# Keep wind speed (u10 = eastward wind component m/s)
# Convert to absolute wind speed
df_raw['wind_speed'] = df_raw['u10'].abs()

# Drop original Kelvin columns
df_raw = df_raw.drop(columns=['t2m', 'd2m', 'u10'])

# Extract time fields
df_raw['valid_time'] = pd.to_datetime(df_raw['valid_time'])
df_raw['year']       = df_raw['valid_time'].dt.year
df_raw['month']      = df_raw['valid_time'].dt.month

# Aggregate to national monthly averages
df_weather_v2 = df_raw.groupby(['year', 'month']).agg(
    temperature_c = ('temperature_c', 'mean'),
    humidity_pct  = ('humidity_pct',  'mean'),
    wind_speed    = ('wind_speed',    'mean'),
).reset_index()

print("✅ New weather dataset shape:", df_weather_v2.shape)
print("Years:", df_weather_v2['year'].min(), "to", df_weather_v2['year'].max())
print(df_weather_v2.head(12))

# Save
df_weather_v2.to_csv('data/processed/era5_nigeria_v2_monthly.csv', index=False)
print("✅ Saved to data/processed/era5_nigeria_v2_monthly.csv")

✅ New weather dataset shape: (220, 5)
Years: 2008 to 2026
    year  month  temperature_c  humidity_pct  wind_speed
0   2008      1      23.870647     35.920704    1.263670
1   2008      2      25.802450     29.349182    1.775623
2   2008      3      28.990774     40.332932    1.191437
3   2008      4      28.880919     49.625439    0.864664
4   2008      5      28.485775     62.815266    0.713356
5   2008      6      27.099874     71.180832    1.055518
6   2008      7      25.390240     78.898163    1.554146
7   2008      8      24.813053     82.868454    1.367703
8   2008      9      25.605160     80.295982    0.941896
9   2008     10      26.397943     66.530289    0.778065
10  2008     11      26.207600     48.486927    1.297227
11  2008     12      25.674116     44.154854    1.146068
✅ Saved to data/processed/era5_nigeria_v2_monthly.csv


In [36]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/progressive/Documents/Courses/Final-Year-Disease-Prediction')
print("✅ Libraries loaded successfully")
print(f"   PyTorch version: {torch.__version__}")

✅ Libraries loaded successfully
   PyTorch version: 2.12.0


In [41]:
import os
os.chdir('/Users/progressive/Documents/Courses/Final-Year-Disease-Prediction')
print("✅ Directory:", os.getcwd())

✅ Directory: /Users/progressive/Documents/Courses/Final-Year-Disease-Prediction


In [45]:
# ── 1. Load and prepare data ───────────────────────────────────────
df_weather = pd.read_csv('data/processed/era5_nigeria_v2_monthly.csv')
df_lassa_raw = pd.read_csv('data/raw/clean_lassa_fever_map_data.csv')

# Aggregate Lassa nationally by year
df_lassa = df_lassa_raw.groupby('year').agg(
    confirmed_cases = ('confirmed', 'sum'),
    deaths          = ('deaths',    'sum'),
).reset_index()

# Merge weather + lassa on year
df = pd.merge(df_weather, df_lassa, on='year', how='inner')
print("✅ Merged shape:", df.shape)
print("Years:", df['year'].min(), "to", df['year'].max())

# ── 2. Create target variable ──────────────────────────────────────
df['rainy_season'] = df['month'].between(5, 10).astype(int)
median_cases = df['confirmed_cases'].median()
df['high_severity_year'] = (df['confirmed_cases'] > median_cases).astype(int)
df['high_risk'] = (
    (df['rainy_season'] == 1) &
    (df['high_severity_year'] == 1)
).astype(int)

print(f"✅ Target — High risk: {df['high_risk'].sum()}, Low risk: {(df['high_risk']==0).sum()}")

# ── 3. Add lag features ────────────────────────────────────────────
df = df.sort_values(['year','month']).reset_index(drop=True)
df['temp_lag1']     = df['temperature_c'].shift(1)
df['humidity_lag1'] = df['humidity_pct'].shift(1)
df['temp_lag2']     = df['temperature_c'].shift(2)
df['humidity_lag2'] = df['humidity_pct'].shift(2)
df['wind_lag1']     = df['wind_speed'].shift(1)
df = df.dropna().reset_index(drop=True)
print("✅ Final shape after lagging:", df.shape)

# ── 4. Define feature groups ───────────────────────────────────────
# Group 1 — direct weather (for neural network input 1)
weather_features = [
    'temperature_c', 'humidity_pct', 'wind_speed',
    'temp_lag1', 'humidity_lag1', 'wind_lag1',
]
# Group 2 — seasonal features (for neural network input 2)
seasonal_features = [
    'month', 'rainy_season',
    'temp_lag2', 'humidity_lag2',
]
# All features combined (for traditional models)
all_features = weather_features + seasonal_features

X_all      = df[all_features]
X_weather  = df[weather_features]
X_seasonal = df[seasonal_features]
y          = df['high_risk']

print("\n✅ Feature groups:")
print(f"   Weather features:  {weather_features}")
print(f"   Seasonal features: {seasonal_features}")
print(f"   Class distribution: {y.value_counts().to_dict()}")

# ── 5. Scale features ──────────────────────────────────────────────
scaler_all      = StandardScaler()
scaler_weather  = StandardScaler()
scaler_seasonal = StandardScaler()

X_all_scaled      = scaler_all.fit_transform(X_all)
X_weather_scaled  = scaler_weather.fit_transform(X_weather)
X_seasonal_scaled = scaler_seasonal.fit_transform(X_seasonal)

print("✅ Features scaled")

# ── 6. Cross validation setup ──────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── 7. Traditional models ──────────────────────────────────────────
traditional_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=4,
                               random_state=42, class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=200, max_depth=3,
                               learning_rate=0.1, random_state=42,
                               scale_pos_weight=len(y[y==0])/len(y[y==1]),
                               eval_metric='logloss', verbosity=0),
}

print("\n=== TRADITIONAL MODELS (5-Fold Stratified CV) ===")
trad_results = {}
for name, model in traditional_models.items():
    acc = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='accuracy')
    f1  = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='f1')
    rec = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='recall')
    pre = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='precision')
    trad_results[name] = {
        'Accuracy': acc.mean(), 'F1': f1.mean(),
        'Recall': rec.mean(), 'Precision': pre.mean()
    }
    print(f"\n{name}:")
    print(f"  Accuracy:  {acc.mean():.2%} (+/- {acc.std():.2%})")
    print(f"  F1 Score:  {f1.mean():.2%} (+/- {f1.std():.2%})")
    print(f"  Recall:    {rec.mean():.2%} (+/- {rec.std():.2%})")
    print(f"  Precision: {pre.mean():.2%} (+/- {pre.std():.2%})")

# ── 8. Multi-Input Neural Network (PyTorch) ────────────────────────
print("\n=== MULTI-INPUT NEURAL NETWORK ===")

class MultiInputNet(nn.Module):
    def __init__(self, weather_dim, seasonal_dim):
        super().__init__()
        # Branch 1 — processes weather features
        self.weather_branch = nn.Sequential(
            nn.Linear(weather_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
        )
        # Branch 2 — processes seasonal features
        self.seasonal_branch = nn.Sequential(
            nn.Linear(seasonal_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 8),
            nn.ReLU(),
        )
        # Combined layer — merges both branches
        self.combined = nn.Sequential(
            nn.Linear(16 + 8, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x_weather, x_seasonal):
        w = self.weather_branch(x_weather)
        s = self.seasonal_branch(x_seasonal)
        combined = torch.cat([w, s], dim=1)
        return self.combined(combined).squeeze()

# Train neural network with cross validation
nn_fold_scores = {'accuracy':[], 'f1':[], 'recall':[], 'precision':[]}

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

for fold, (train_idx, val_idx) in enumerate(cv.split(X_weather_scaled, y)):
    # Prepare tensors
    Xw_train = torch.FloatTensor(X_weather_scaled[train_idx])
    Xs_train = torch.FloatTensor(X_seasonal_scaled[train_idx])
    y_train  = torch.FloatTensor(y.values[train_idx])

    Xw_val   = torch.FloatTensor(X_weather_scaled[val_idx])
    Xs_val   = torch.FloatTensor(X_seasonal_scaled[val_idx])
    y_val    = y.values[val_idx]

    # Init model
    model_nn = MultiInputNet(
        weather_dim  = len(weather_features),
        seasonal_dim = len(seasonal_features)
    )

    # Handle class imbalance
    pos_weight = torch.tensor([len(y_train[y_train==0]) / max(len(y_train[y_train==1]),1)])
    criterion  = nn.BCELoss()
    optimizer  = optim.Adam(model_nn.parameters(), lr=0.001, weight_decay=1e-4)

    # Train
    model_nn.train()
    for epoch in range(150):
        optimizer.zero_grad()
        outputs = model_nn(Xw_train, Xs_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    # Evaluate
    model_nn.eval()
    with torch.no_grad():
        preds_prob = model_nn(Xw_val, Xs_val).numpy()
        preds = (preds_prob >= 0.5).astype(int)

    nn_fold_scores['accuracy'].append(accuracy_score(y_val, preds))
    nn_fold_scores['f1'].append(f1_score(y_val, preds, zero_division=0))
    nn_fold_scores['recall'].append(recall_score(y_val, preds, zero_division=0))
    nn_fold_scores['precision'].append(precision_score(y_val, preds, zero_division=0))
    print(f"  Fold {fold+1}: Acc={nn_fold_scores['accuracy'][-1]:.2%} F1={nn_fold_scores['f1'][-1]:.2%} Recall={nn_fold_scores['recall'][-1]:.2%}")

nn_results = {k: np.mean(v) for k, v in nn_fold_scores.items()}
print(f"\nNeural Network Mean Results:")
print(f"  Accuracy:  {nn_results['accuracy']:.2%}")
print(f"  F1 Score:  {nn_results['f1']:.2%}")
print(f"  Recall:    {nn_results['recall']:.2%}")
print(f"  Precision: {nn_results['precision']:.2%}")

# ── 9. Ensemble model ──────────────────────────────────────────────
print("\n=== ENSEMBLE MODEL (Soft Voting) ===")

# Train all base models on full data first
lr_final  = LogisticRegression(max_iter=1000, class_weight='balanced')
rf_final  = RandomForestClassifier(n_estimators=200, max_depth=4,
                random_state=42, class_weight='balanced')
xgb_final = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                random_state=42, scale_pos_weight=len(y[y==0])/len(y[y==1]),
                eval_metric='logloss', verbosity=0)

ensemble = VotingClassifier(
    estimators=[
        ('lr',  lr_final),
        ('rf',  rf_final),
        ('xgb', xgb_final),
    ],
    voting='soft'
)

ens_acc = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='accuracy')
ens_f1  = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='f1')
ens_rec = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='recall')
ens_pre = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='precision')

print(f"\nEnsemble (LR + RF + XGBoost):")
print(f"  Accuracy:  {ens_acc.mean():.2%} (+/- {ens_acc.std():.2%})")
print(f"  F1 Score:  {ens_f1.mean():.2%} (+/- {ens_f1.std():.2%})")
print(f"  Recall:    {ens_rec.mean():.2%} (+/- {ens_rec.std():.2%})")
print(f"  Precision: {ens_pre.mean():.2%} (+/- {ens_pre.std():.2%})")

# ── 10. Final comparison table ─────────────────────────────────────
print("\n" + "="*60)
print("FULL MODEL COMPARISON")
print("="*60)
print(f"{'Model':<30} {'Accuracy':>10} {'F1':>10} {'Recall':>10}")
print("-"*60)
for name, res in trad_results.items():
    print(f"{name:<30} {res['Accuracy']:>10.2%} {res['F1']:>10.2%} {res['Recall']:>10.2%}")
print(f"{'Neural Network':<30} {nn_results['accuracy']:>10.2%} {nn_results['f1']:>10.2%} {nn_results['recall']:>10.2%}")
print(f"{'Ensemble (LR+RF+XGB)':<30} {ens_acc.mean():>10.2%} {ens_f1.mean():>10.2%} {ens_rec.mean():>10.2%}")
print("="*60)

# ── 11. Save best model ────────────────────────────────────────────
import joblib
ensemble.fit(X_all_scaled, y)
joblib.dump(ensemble,    'outputs/ensemble_model.pkl')
joblib.dump(scaler_all,  'outputs/scaler_v2.pkl')
print("\n✅ Ensemble model saved!")

✅ Merged shape: (72, 7)
Years: 2020 to 2025
✅ Target — High risk: 18, Low risk: 54
✅ Final shape after lagging: (70, 15)

✅ Feature groups:
   Weather features:  ['temperature_c', 'humidity_pct', 'wind_speed', 'temp_lag1', 'humidity_lag1', 'wind_lag1']
   Seasonal features: ['month', 'rainy_season', 'temp_lag2', 'humidity_lag2']
   Class distribution: {0: 52, 1: 18}
✅ Features scaled

=== TRADITIONAL MODELS (5-Fold Stratified CV) ===

Logistic Regression:
  Accuracy:  74.29% (+/- 12.45%)
  F1 Score:  64.55% (+/- 16.00%)
  Recall:    88.33% (+/- 14.53%)
  Precision: 52.76% (+/- 17.71%)

Random Forest:
  Accuracy:  68.57% (+/- 5.71%)
  F1 Score:  38.00% (+/- 19.39%)
  Recall:    46.67% (+/- 24.49%)
  Precision: 33.71% (+/- 18.62%)

XGBoost:
  Accuracy:  68.57% (+/- 7.28%)
  F1 Score:  43.71% (+/- 10.60%)
  Recall:    50.00% (+/- 19.00%)
  Precision: 50.71% (+/- 26.76%)

=== MULTI-INPUT NEURAL NETWORK ===
  Fold 1: Acc=71.43% F1=50.00% Recall=66.67%
  Fold 2: Acc=78.57% F1=57.14% Recall=6

In [46]:
df_cholera = pd.read_csv('data/raw/nigeria_cholera.csv')
print("Shape:", df_cholera.shape)
print("Columns:", df_cholera.columns.tolist())
print(df_cholera.head())

Shape: (16, 2)
Columns: ['year', 'cholera_cases']
   year  cholera_cases
0  2010        41787.0
1  2011        23366.0
2  2012          597.0
3  2013         6600.0
4  2014        35996.0


In [47]:
print("="*60)
print("FULL MODEL COMPARISON SUMMARY")
print("="*60)
print(f"{'Model':<30} {'Accuracy':>10} {'F1':>10} {'Recall':>10}")
print("-"*60)
for name, res in trad_results.items():
    print(f"{name:<30} {res['Accuracy']:>10.2%} {res['F1']:>10.2%} {res['Recall']:>10.2%}")
print(f"{'Neural Network':<30} {nn_results['accuracy']:>10.2%} {nn_results['f1']:>10.2%} {nn_results['recall']:>10.2%}")
print(f"{'Ensemble (LR+RF+XGB)':<30} {ens_acc.mean():>10.2%} {ens_f1.mean():>10.2%} {ens_rec.mean():>10.2%}")
print("="*60)

FULL MODEL COMPARISON SUMMARY
Model                            Accuracy         F1     Recall
------------------------------------------------------------
Logistic Regression                74.29%     64.55%     88.33%
Random Forest                      68.57%     38.00%     46.67%
XGBoost                            68.57%     43.71%     50.00%
Neural Network                     72.86%     36.43%     41.67%
Ensemble (LR+RF+XGB)               68.57%     47.27%     56.67%


In [48]:
print(df_cholera)

    year  cholera_cases
0   2010        41787.0
1   2011        23366.0
2   2012          597.0
3   2013         6600.0
4   2014        35996.0
5   2015         5290.0
6   2016          768.0
7   2017         1956.0
8   2018         3640.0
9   2019        30407.0
10  2020         2558.0
11  2021       112746.0
12  2022        48574.0
13  2023        18088.0
14  2024         8173.0
15  2025         5000.0


In [52]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score, recall_score, precision_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/progressive/Documents/Courses/Final-Year-Disease-Prediction')

# ── 1. Load all data ───────────────────────────────────────────────
df_weather  = pd.read_csv('data/processed/era5_nigeria_v2_monthly.csv')
df_lassa    = pd.read_csv('data/raw/clean_lassa_fever_map_data.csv')
df_cholera  = pd.read_csv('data/raw/nigeria_cholera.csv')

# ── 2. Aggregate Lassa nationally ─────────────────────────────────
df_lassa_annual = df_lassa.groupby('year').agg(
    lassa_cases = ('confirmed', 'sum'),
).reset_index()

# ── 3. Merge all three datasets on year ───────────────────────────
df = pd.merge(df_weather,     df_lassa_annual, on='year', how='inner')
df = pd.merge(df,             df_cholera,      on='year', how='inner')

print("✅ Merged shape:", df.shape)
print("Years:", df['year'].min(), "to", df['year'].max())
print("Columns:", df.columns.tolist())

# ── 4. Create combined outbreak risk target ────────────────────────
# High risk = EITHER Lassa OR Cholera is above its median in that year
# AND it is rainy season (May-Oct)
df['rainy_season'] = df['month'].between(5, 10).astype(int)

median_lassa   = df['lassa_cases'].median()
median_cholera = df['cholera_cases'].median()

df['lassa_high']   = (df['lassa_cases']   > median_lassa).astype(int)
df['cholera_high'] = (df['cholera_cases'] > median_cholera).astype(int)

# Combined: high risk if either disease is in high severity year
# AND it is rainy season
df['any_disease_high'] = ((df['lassa_high'] == 1) | (df['cholera_high'] == 1)).astype(int)
df['high_risk'] = (
    (df['rainy_season'] == 1) &
    (df['any_disease_high'] == 1)
).astype(int)

print(f"\n✅ Target distribution:")
print(f"   High risk months: {df['high_risk'].sum()}")
print(f"   Low risk months:  {(df['high_risk']==0).sum()}")
print(f"   Lassa median:     {median_lassa:.0f} cases/year")
print(f"   Cholera median:   {median_cholera:.0f} cases/year")

# ── 5. Add lag features ────────────────────────────────────────────
df = df.sort_values(['year','month']).reset_index(drop=True)
df['temp_lag1']     = df['temperature_c'].shift(1)
df['humidity_lag1'] = df['humidity_pct'].shift(1)
df['temp_lag2']     = df['temperature_c'].shift(2)
df['humidity_lag2'] = df['humidity_pct'].shift(2)
df['wind_lag1']     = df['wind_speed'].shift(1)
df = df.dropna().reset_index(drop=True)

print(f"\n✅ Final dataset shape: {df.shape}")

# ── 6. Feature groups ──────────────────────────────────────────────
weather_features  = ['temperature_c','humidity_pct','wind_speed',
                     'temp_lag1','humidity_lag1','wind_lag1']
seasonal_features = ['month','rainy_season','temp_lag2','humidity_lag2']
all_features      = weather_features + seasonal_features

X_all      = df[all_features].values
X_weather  = df[weather_features].values
X_seasonal = df[seasonal_features].values
y          = df['high_risk']

print(f"✅ Class distribution: {y.value_counts().to_dict()}")

# ── 7. Scale features ──────────────────────────────────────────────
scaler_all      = StandardScaler()
scaler_weather  = StandardScaler()
scaler_seasonal = StandardScaler()

X_all_scaled      = scaler_all.fit_transform(X_all)
X_weather_scaled  = scaler_weather.fit_transform(X_weather)
X_seasonal_scaled = scaler_seasonal.fit_transform(X_seasonal)

# ── 8. Cross validation ────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── 9. Traditional models ──────────────────────────────────────────
traditional_models = {
    'Logistic Regression': LogisticRegression(
                               max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(
                               n_estimators=200, max_depth=4,
                               random_state=42, class_weight='balanced'),
    'XGBoost':             XGBClassifier(
                               n_estimators=200, max_depth=3,
                               learning_rate=0.1, random_state=42,
                               scale_pos_weight=len(y[y==0])/len(y[y==1]),
                               eval_metric='logloss', verbosity=0),
}

print("\n=== TRADITIONAL MODELS ===")
trad_results = {}
for name, model in traditional_models.items():
    acc = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='accuracy')
    f1  = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='f1')
    rec = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='recall')
    pre = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='precision')
    trad_results[name] = {
        'Accuracy': acc.mean(), 'F1': f1.mean(),
        'Recall': rec.mean(), 'Precision': pre.mean()
    }
    print(f"\n{name}:")
    print(f"  Accuracy:  {acc.mean():.2%} (+/- {acc.std():.2%})")
    print(f"  F1 Score:  {f1.mean():.2%} (+/- {f1.std():.2%})")
    print(f"  Recall:    {rec.mean():.2%} (+/- {rec.std():.2%})")
    print(f"  Precision: {pre.mean():.2%} (+/- {pre.std():.2%})")

# ── 10. Multi-Input Neural Network ────────────────────────────────
print("\n=== MULTI-INPUT NEURAL NETWORK ===")

class MultiInputNet(nn.Module):
    def __init__(self, weather_dim, seasonal_dim):
        super().__init__()
        self.weather_branch = nn.Sequential(
            nn.Linear(weather_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),          nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),          nn.ReLU(),
        )
        self.seasonal_branch = nn.Sequential(
            nn.Linear(seasonal_dim, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16),           nn.ReLU(),
        )
        self.combined = nn.Sequential(
            nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),  nn.Sigmoid()
        )
    def forward(self, xw, xs):
        return self.combined(
            torch.cat([self.weather_branch(xw),
                       self.seasonal_branch(xs)], dim=1)
        ).squeeze()

nn_scores = {'accuracy':[],'f1':[],'recall':[],'precision':[]}

for fold,(train_idx,val_idx) in enumerate(cv.split(X_weather_scaled, y)):
    Xw_tr = torch.FloatTensor(X_weather_scaled[train_idx])
    Xs_tr = torch.FloatTensor(X_seasonal_scaled[train_idx])
    y_tr  = torch.FloatTensor(y.values[train_idx])
    Xw_v  = torch.FloatTensor(X_weather_scaled[val_idx])
    Xs_v  = torch.FloatTensor(X_seasonal_scaled[val_idx])
    y_v   = y.values[val_idx]

    net = MultiInputNet(len(weather_features), len(seasonal_features))
    opt = optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-4)
    crit = nn.BCELoss()

    net.train()
    for _ in range(200):
        opt.zero_grad()
        loss = crit(net(Xw_tr, Xs_tr), y_tr)
        loss.backward()
        opt.step()

    net.eval()
    with torch.no_grad():
        preds = (net(Xw_v, Xs_v).numpy() >= 0.5).astype(int)

    nn_scores['accuracy'].append(accuracy_score(y_v, preds))
    nn_scores['f1'].append(f1_score(y_v, preds, zero_division=0))
    nn_scores['recall'].append(recall_score(y_v, preds, zero_division=0))
    nn_scores['precision'].append(precision_score(y_v, preds, zero_division=0))
    print(f"  Fold {fold+1}: Acc={nn_scores['accuracy'][-1]:.2%} "
          f"F1={nn_scores['f1'][-1]:.2%} "
          f"Recall={nn_scores['recall'][-1]:.2%}")

nn_results = {k: np.mean(v) for k,v in nn_scores.items()}
print(f"\n  Mean — Accuracy: {nn_results['accuracy']:.2%} "
      f"F1: {nn_results['f1']:.2%} "
      f"Recall: {nn_results['recall']:.2%}")

# ── 11. Ensemble ───────────────────────────────────────────────────
print("\n=== ENSEMBLE MODEL ===")
ensemble = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf',  RandomForestClassifier(n_estimators=200, max_depth=4,
                    random_state=42, class_weight='balanced')),
        ('xgb', XGBClassifier(n_estimators=200, max_depth=3,
                    learning_rate=0.1, random_state=42,
                    scale_pos_weight=len(y[y==0])/len(y[y==1]),
                    eval_metric='logloss', verbosity=0)),
    ], voting='soft'
)
ens_acc = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='accuracy')
ens_f1  = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='f1')
ens_rec = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='recall')
ens_pre = cross_val_score(ensemble, X_all_scaled, y, cv=cv, scoring='precision')
print(f"  Accuracy:  {ens_acc.mean():.2%}")
print(f"  F1 Score:  {ens_f1.mean():.2%}")
print(f"  Recall:    {ens_rec.mean():.2%}")
print(f"  Precision: {ens_pre.mean():.2%}")

# ── 12. Final comparison ───────────────────────────────────────────
print("\n" + "="*65)
print("FINAL MODEL COMPARISON — Lassa + Cholera Combined Target")
print("="*65)
print(f"{'Model':<32} {'Accuracy':>10} {'F1':>10} {'Recall':>10}")
print("-"*65)
for name, res in trad_results.items():
    print(f"{name:<32} {res['Accuracy']:>10.2%} "
          f"{res['F1']:>10.2%} {res['Recall']:>10.2%}")
print(f"{'Neural Network (Multi-Input)':<32} "
      f"{nn_results['accuracy']:>10.2%} "
      f"{nn_results['f1']:>10.2%} "
      f"{nn_results['recall']:>10.2%}")
print(f"{'Ensemble (LR+RF+XGB)':<32} "
      f"{ens_acc.mean():>10.2%} "
      f"{ens_f1.mean():>10.2%} "
      f"{ens_rec.mean():>10.2%}")
print("="*65)

# ── 13. Save models ────────────────────────────────────────────────
import joblib
os.makedirs('outputs', exist_ok=True)
ensemble.fit(X_all_scaled, y)
joblib.dump(ensemble,   'outputs/ensemble_model_v2.pkl')
joblib.dump(scaler_all, 'outputs/scaler_v2.pkl')
print("\n✅ Models saved!")
print(f"✅ Dataset size: {len(df)} monthly observations")
print(f"✅ Diseases:     Lassa Fever + Cholera")
print(f"✅ Period:       {df['year'].min()} to {df['year'].max()}")



✅ Merged shape: (72, 7)
Years: 2020 to 2025
Columns: ['year', 'month', 'temperature_c', 'humidity_pct', 'wind_speed', 'lassa_cases', 'cholera_cases']

✅ Target distribution:
   High risk months: 30
   Low risk months:  42
   Lassa median:     1168 cases/year
   Cholera median:   13130 cases/year

✅ Final dataset shape: (70, 17)
✅ Class distribution: {0: 40, 1: 30}

=== TRADITIONAL MODELS ===

Logistic Regression:
  Accuracy:  91.43% (+/- 8.33%)
  F1 Score:  91.60% (+/- 7.88%)
  Recall:    100.00% (+/- 0.00%)
  Precision: 85.48% (+/- 13.31%)

Random Forest:
  Accuracy:  88.57% (+/- 9.69%)
  F1 Score:  88.05% (+/- 9.96%)
  Recall:    93.33% (+/- 8.16%)
  Precision: 84.29% (+/- 13.93%)

XGBoost:
  Accuracy:  84.29% (+/- 13.09%)
  F1 Score:  82.16% (+/- 14.70%)
  Recall:    83.33% (+/- 14.91%)
  Precision: 81.83% (+/- 15.92%)

=== MULTI-INPUT NEURAL NETWORK ===
  Fold 1: Acc=78.57% F1=72.73% Recall=66.67%
  Fold 2: Acc=64.29% F1=54.55% Recall=50.00%
  Fold 3: Acc=78.57% F1=80.00% Recall=10

In [51]:

print("="*65)
print("FULL MODEL COMPARISON — Lassa + Cholera (2020-2025)")
print("="*65)
print(f"{'Model':<32} {'Accuracy':>10} {'F1':>10} {'Recall':>10}")
print("-"*65)
for name, res in trad_results.items():
    print(f"{name:<32} {res['Accuracy']:>10.2%} {res['F1']:>10.2%} {res['Recall']:>10.2%}")
print(f"{'Neural Network (Multi-Input)':<32} {nn_results['accuracy']:>10.2%} {nn_results['f1']:>10.2%} {nn_results['recall']:>10.2%}")
print(f"{'Ensemble (LR+RF+XGB)':<32} {ens_acc.mean():>10.2%} {ens_f1.mean():>10.2%} {ens_rec.mean():>10.2%}")
print("="*65)
print(f"\nDataset: {len(df)} monthly observations")
print(f"Period:  {df['year'].min()} to {df['year'].max()}")
print(f"Diseases: Lassa Fever + Cholera")
print(f"High risk months: {y.sum()} of {len(y)}")

FULL MODEL COMPARISON — Lassa + Cholera (2020-2025)
Model                              Accuracy         F1     Recall
-----------------------------------------------------------------
Logistic Regression                  91.43%     91.60%    100.00%
Random Forest                        88.57%     88.05%     93.33%
XGBoost                              84.29%     82.16%     83.33%
Neural Network (Multi-Input)         82.86%     79.64%     80.00%
Ensemble (LR+RF+XGB)                 88.57%     88.10%     93.33%

Dataset: 70 monthly observations
Period:  2020 to 2025
Diseases: Lassa Fever + Cholera
High risk months: 30 of 70
